## Orquestrador Contínuo: Raw ➔ Bronze ➔ Silver ➔ Gold

Notebook mestre do polling e da execução em cadeia das camadas:
- **Bronze** (`04_ingestao_bronze`) é executada a cada checagem e consome os arquivos novos da **Raw**.
- **Silver** (`05_ingestao_silver`) consome a **Bronze** e só roda quando a Bronze foi atualizada ou quando ficaram micro-lotes pendentes (falha anterior). Também grava a `dq_monitoring_logs`.
- **Gold** (`06_ingestao_gold`) consome a **Silver** e a `dq_monitoring_logs` e só roda quando a Silver/log foram atualizados ou quando a última execução da Gold falhou.

**Polling com backoff** (configurável em `POLITICA_POLLING`):

| Nível | Intervalo | Tentativas sem dados novos |
|-------|-----------|----------------------------|
| 1 | 30 s | 3 |
| 2 | 1 min | 3 |
| 3 | 5 min | 3 |
| 4 | 10 min | 3 |
| 5 | 30 min | 3 |
| 6 | 1 h | 3 |
| 7 | 2 h | contínuo |

Sempre que a Bronze ingere um arquivo novo, a política reinicia no nível 1. O loop roda até ser interrompido manualmente.

In [0]:
import json
import time
from datetime import datetime, timedelta

caminho_bronze = "04_ingestao_bronze"
caminho_silver = "05_ingestao_silver"
caminho_gold   = "06_ingestao_gold"
TIMEOUT_NOTEBOOK_S = 1200

# Política de polling com backoff: (intervalo em segundos, tentativas sem dados novos nesse nível).
# Esgotadas as tentativas, passa ao próximo nível; o último (None) se repete indefinidamente.
# Sempre que a Bronze ingere arquivo novo, a contagem volta ao primeiro nível.
POLITICA_POLLING = [
    (30,       3),
    (60,       3),
    (5 * 60,   3),
    (10 * 60,  3),
    (30 * 60,  3),
    (60 * 60,  3),
    (120 * 60, None),
]

def formatar_intervalo(segundos):
    if segundos < 60:
        return f"{segundos}s"
    if segundos < 3600:
        return f"{segundos // 60}min"
    return f"{segundos // 3600}h"

def executar_notebook(caminho):
    """Executa o notebook filho e devolve o JSON retornado por dbutils.notebook.exit."""
    retorno = dbutils.notebook.run(caminho, timeout_seconds=TIMEOUT_NOTEBOOK_S)
    return json.loads(retorno) if retorno else {}

In [0]:
nivel, tentativa = 0, 0      # posição atual na POLITICA_POLLING
silver_pendente = True       # na 1ª rodada a Silver roda para recuperar lotes que ficaram para trás
gold_pendente = True         # idem para a Gold

print("Iniciando orquestração Raw ➔ Bronze ➔ Silver ➔ Gold com polling por backoff...\n")

try:
    while True:
        print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] ─── Verificando novos dados na Raw ───")

        # 1. Bronze consome a Raw
        try:
            res_bronze = executar_notebook(caminho_bronze)
            qtd_bronze = res_bronze.get("qtd_ingeridos", 0)
            if qtd_bronze > 0:
                print(f"  ✔ Bronze ingeriu {qtd_bronze} novo(s) arquivo(s):")
                for arq in res_bronze.get("arquivos", []):
                    print(f"     └─ {arq}")
            else:
                print("  ℹ Bronze: nenhum arquivo novo ingerido.")
            for arq in res_bronze.get("falhas", []):
                print(f"     ⚠ Falha na ingestão (será tentado de novo): {arq}")
        except Exception as e:
            print(f"  ✖ Erro na execução da Bronze: {e}")
            qtd_bronze = 0

        # 2. Silver consome a Bronze: roda se a Bronze foi atualizada ou se ficaram lotes pendentes
        silver_atualizada = False
        if qtd_bronze > 0 or silver_pendente:
            print("  ▶ Executando Silver sobre os micro-lotes pendentes da Bronze...")
            try:
                res_silver = executar_notebook(caminho_silver)
                silver_pendente = res_silver.get("qtd_pendentes", 0) > 0 or res_silver.get("logs_pendentes", False)
                silver_atualizada = res_silver.get("qtd_processados", 0) > 0 or res_silver.get("qtd_linhas_log", 0) > 0
                print(f"  ✔ Silver processou {res_silver.get('qtd_processados', 0)} micro-lote(s).")
                for arq in res_silver.get("falhas", []):
                    print(f"     ⚠ Pendente (será tentado de novo): {arq}")
                if res_silver.get("logs_pendentes"):
                    print("     ⚠ dq_monitoring_logs não gravado (será tentado de novo).")
            except Exception as e:
                print(f"  ✖ Erro na execução da Silver: {e}")
                silver_pendente = True
        else:
            print("  ⏭ Silver ignorada: Bronze sem atualização e nenhum lote pendente.")

        # 3. Gold consome a Silver e o log: roda se foram atualizados ou se a última Gold falhou
        if silver_atualizada or gold_pendente:
            print("  ▶ Executando Gold (métricas de DQ para o BI)...")
            try:
                res_gold = executar_notebook(caminho_gold)
                gold_pendente = bool(res_gold.get("falhas"))
                for nome, qtd in res_gold.get("sql", {}).items():
                    print(f"  ✔ Gold {nome}: {qtd} linha(s) no SQL Server.")
                for nome in res_gold.get("falhas", []):
                    print(f"     ⚠ Gold {nome} com falha (será tentada de novo).")
            except Exception as e:
                print(f"  ✖ Erro na execução da Gold: {e}")
                gold_pendente = True
        else:
            print("  ⏭ Gold ignorada: Silver e log sem atualização.")

        # 4. Backoff: dado novo reinicia a política; sem dado novo, avança na escada de intervalos
        if qtd_bronze > 0:
            nivel, tentativa = 0, 0

        intervalo, max_tentativas = POLITICA_POLLING[nivel]
        tentativa += 1
        desc_tentativa = f"{tentativa}/{max_tentativas}" if max_tentativas else f"{tentativa} (contínuo)"
        proxima = datetime.now() + timedelta(seconds=intervalo)
        print(f"─── Nível {nivel + 1}/{len(POLITICA_POLLING)}, tentativa {desc_tentativa}: "
              f"próxima checagem em {formatar_intervalo(intervalo)} ({proxima:%H:%M:%S}) ───\n")

        if max_tentativas and tentativa >= max_tentativas and nivel < len(POLITICA_POLLING) - 1:
            nivel, tentativa = nivel + 1, 0

        time.sleep(intervalo)

except KeyboardInterrupt:
    print("\nOrquestrador interrompido manualmente pelo usuário.")